# 03 - Entrenar modelo con transfer learning

Objetivo: entrenar un clasificador de cuatro clases y guardar el mejor modelo.

In [ ]:
from pathlib import Path
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'dataset_augmented'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASSES = ['arbejas', 'arroz', 'frijol', 'maiz_pira']

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=CLASSES
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=CLASSES
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    MODEL_DIR / 'modelo_semillas_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

# history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[checkpoint])

In [ ]:
# Fine tuning opcional
# base_model.trainable = True
# for layer in base_model.layers[:-30]:
#     layer.trainable = False
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
# fine_history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[checkpoint])